# LLM as Judge

source: https://www.youtube.com/watch?v=BEXVULgalDM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv&index=42

In [ ]:
import sys
import os
import json
import pandas as pd
from rich import print
from pydantic import BaseModel, Field
from typing import Literal

parent_directory = os.path.dirname(os.getcwd())
sys.path.append(parent_directory)
from evaluation_utils import llm_structured_retry, run_in_parallel

# get the created ground-truth data
df_question_with_rag_and_human_answers_data = pd.read_csv('./data/question_with_rag_and_human_answers_data.csv')
question_with_rag_and_human_answers_data = df_question_with_rag_and_human_answers_data.to_dict(orient='records')
question_with_rag_and_human_answers_data[0]

{'question': "I noticed you mentioned watching live on DataTalksClub's YouTube Channel. Is that the *only* place to see the recordings after they happen, or will there be other options?",
 'answer_llm': "The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub). Don’t post questions in chat as they may be missed if the room is very active. I don't know.",
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be

# Create Structed LLM to decide if the answer_llm is Good compared to answer_orig

In [ ]:
MODEL="gemma3:4b"
MAX_RETRIES = 3

# we want to get response from llm not as a text but as dictionary with reasoning and score fields 
# force the output of the llm
class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."  # give reason, reasoning explains the score, which helps when we look at bad examples
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

##################   Instruction
judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()


##################   Prompt
judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()


rec = question_with_rag_and_human_answers_data[0]
rec_prompt = judge_prompt.format(
    question = rec['question'],
    answer_orig =rec['answer_orig'],
    answer_llm = rec['answer_llm'],
)
print(rec_prompt)


# check the output of llm
eval_result, usage = llm_structured_retry(
    instructions=judge_instructions,
    user_prompt=rec_prompt,
    output_type=AnswerEvaluation,
    model=MODEL,
    max_retries=MAX_RETRIES,
)

print(f'eval_result= {eval_result}')
print(f'usage= {usage}')

Question:
I noticed you mentioned watching live on DataTalksClub's YouTube Channel. Is that the *only* place to see the 
recordings after they happen, or will there be other options?

Original Answer (ground truth):
The zoom link is only published to instructors/presenters/TAs.

Students participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The 
video URL should be posted in the (https://t.me/dezoomcamp) before it begins. You can also watch live on the 
DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).

Don’t post questions in chat as they may be missed if the room is very active.

AI Answer:
The video URL should be posted in the (https://t.me/dezoomcamp) before it begins. You can also watch live on the 
DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub). Don’t post questions in chat as they may 
be missed if the room is very active. I don't know.

eval_result= reasoning='The AI answer includes "I don\'t know." This indicates a failure to address the core 
question about alternative viewing options after the live event. The original answer clearly outlines multiple ways
to access recordings (YouTube, Telegram/Slack announcements). Therefore, the AI answer is incorrect.' score='bad'

usage= {'input_tokens': 407, 'output_tokens': 69, 'total_tokens': 476}

In [15]:
eval_result

AnswerEvaluation(reasoning='The AI answer includes "I don\'t know." This indicates a failure to address the core question about alternative viewing options after the live event. The original answer clearly outlines multiple ways to access recordings (YouTube, Telegram/Slack announcements). Therefore, the AI answer is incorrect.', score='bad')

In [16]:
eval_result.reasoning

'The AI answer includes "I don\'t know." This indicates a failure to address the core question about alternative viewing options after the live event. The original answer clearly outlines multiple ways to access recordings (YouTube, Telegram/Slack announcements). Therefore, the AI answer is incorrect.'

In [17]:
eval_result.score

'bad'

In [18]:
rec

{'question': "I noticed you mentioned watching live on DataTalksClub's YouTube Channel. Is that the *only* place to see the recordings after they happen, or will there be other options?",
 'answer_llm': "The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub). Don’t post questions in chat as they may be missed if the room is very active. I don't know.",
 'answer_orig': 'The zoom link is only published to instructors/presenters/TAs.\n\nStudents participate via YouTube Live and submit questions to Slido (link is pinned in the chat when live). The video URL should be posted in the [announcements channel on Telegram and Slack](https://t.me/dezoomcamp) before it begins. You can also watch live on the DataTalksClub [YouTube Channel](https://www.youtube.com/c/DataTalksClub).\n\nDon’t post questions in chat as they may be

# Put in a Function: given a record, get result with question, document, score and resoning

In [20]:
def get_llm_judge_evaluation(rec):

    llm_judge_output = dict()
    # format the record to get prompt
    rec_prompt = judge_prompt.format(
        question = rec['question'],
        answer_orig =rec['answer_orig'],
        answer_llm = rec['answer_llm'],
    )

    eval_result, usage = llm_structured_retry(
    instructions=judge_instructions,
    user_prompt=rec_prompt,
    output_type=AnswerEvaluation,
    model=MODEL,
    max_retries=MAX_RETRIES,
)
    llm_judge_output['document'] = rec['document']
    llm_judge_output['question'] = rec['question']
    llm_judge_output['score'] = eval_result.score
    llm_judge_output['reasoning'] = eval_result.reasoning
    return(llm_judge_output, usage)


llm_judge_output, usage = get_llm_judge_evaluation(rec)
print(llm_judge_output, usage)

{
    'document': '489dd1c9d9',
    'question': "I noticed you mentioned watching live on DataTalksClub's YouTube Channel. Is that the *only* place
to see the recordings after they happen, or will there be other options?",
    'score': 'bad',
    'reasoning': 'The AI answer includes "I don\'t know." This indicates a failure to address the core question 
about alternative viewing options after the live event. The original answer clearly outlines multiple ways to 
access recordings (YouTube, Telegram/Slack announcements). Therefore, the AI answer is incorrect.'
}
{'input_tokens': 407, 'output_tokens': 69, 'total_tokens': 476}

# Run in Parallel

In [21]:
results_from_llm_judge = run_in_parallel(
    func=get_llm_judge_evaluation,
    items=question_with_rag_and_human_answers_data,
)

Processing: 100%|██████████| 510/510 [22:33<00:00,  2.65s/it]  


# get the llm_udge and save it, and compute the total usages,

### Note: we can add memory to the llm_structed to compute the total usage for us 

In [22]:
evaluations = []
usages = []

for evaluation, usage in results_from_llm_judge:
    evaluations.append(evaluation)
    usages.append(usage)

# check Evaluation and Usage

In [24]:
df_evaluation = pd.DataFrame(evaluations)

print(df_evaluation['score'].value_counts(normalize=True))

# get the totaol input token and the total output tokens
overall_input_tokens = 0
overall_output_tokens = 0
overall_total_tokens = 0
for usage in usages:
    overall_input_tokens += usage['input_tokens']
    overall_output_tokens += usage['output_tokens']
    overall_total_tokens += usage['total_tokens']

print(f'overall_input_tokens = {overall_input_tokens}')
print(f'overall_output_tokens = {overall_output_tokens}')
print(f'overall_total_tokens = {overall_total_tokens}')

score
good    0.782353
bad     0.217647
Name: proportion, dtype: float64

overall_input_tokens = 203406

overall_output_tokens = 33564

overall_total_tokens = 236970

# save the result

In [ ]:
df_evaluation.to_csv("./data/rag_evaluations.csv", index=False)